```
 텍스트 -> 단어 쪼개기 -> BoW(횟수 카운트) -> TF(비율) -> IDF(희귀도 계산) -> 유사도 계산)
 ```
 ---

In [63]:
!apt-get install default-jdk -y
!pip install konlpy scikit-learn --quiet
print("✅ 완료!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
default-jdk is already the newest version (2:1.11-72build2).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
✅ 완료!


In [64]:
import re          # 정규표현식: 텍스트 정제에 사용
import math        # log() 함수: IDF 계산에 사용

from konlpy.tag import Okt                                    # 한국어 형태소 분석기
from sklearn.feature_extraction.text import TfidfVectorizer   # TF-IDF 자동 계산
from sklearn.metrics.pairwise import cosine_similarity        # 문서 유사도 계산

okt = Okt()

## 1.BoW(Bag of Words) - 단어 빈도 count
단어의 순서는 중요하지 않고, **어떤 단어가 얼마나 자주 나왔는지**

In [65]:
reviewA = "치킨이 맛있어요"
reviewB = "피자가 맛있어요"
print("같나?", reviewA == reviewB)
print("텍스트 비교는 글자가 완전히 일치해야 True, 하나라도 다르면 False")

reviewA_em = [1, 0, 0, 1]   # 치킨O  피자X  배달X  맛있다O
reviewB_em = [0, 1, 0, 1]   # 치킨X  피자O  배달X  맛있다O

print("\n리뷰A:", reviewA_em)
print("리뷰B:", reviewB_em)

같나? False
텍스트 비교는 글자가 완전히 일치해야 True, 하나라도 다르면 False

리뷰A: [1, 0, 0, 1]
리뷰B: [0, 1, 0, 1]


In [66]:
words = ["맛있다", "배달", "빠르다", "서비스", "치킨", "피자", "좋다"]

review1 = "치킨 치킨 배달 맛있다"
review2 = "치킨 피자 배달 빠르다"
review3 = "피자 피자 서비스 좋다"

reviews_BoW = {
    "review1": review1.split(),
    "review2": review2.split(),
    "review3": review3.split()
}

bow_dict = {}

for name, review in reviews_BoW.items():
    bow_dict[name] = [review.count(word) for word in words]

review1_BoW = bow_dict["review1"]
review2_BoW = bow_dict["review2"]
review3_BoW = bow_dict["review3"]

print("어휘: ", words)
print(f"리뷰1: {review1_BoW}  ← 치킨이 2!")
print(f"리뷰2: {review2_BoW}")
print(f"리뷰3: {review3_BoW}  ← 피자가 2!")
print()

어휘:  ['맛있다', '배달', '빠르다', '서비스', '치킨', '피자', '좋다']
리뷰1: [1, 1, 0, 0, 2, 0, 0]  ← 치킨이 2!
리뷰2: [0, 1, 1, 0, 1, 1, 0]
리뷰3: [0, 0, 0, 1, 0, 2, 1]  ← 피자가 2!



### BoW 단점
1) **순서 무시**: "치킨은 좋고 서비스는 나쁘다."랑 "서비스는 좋고 치킨은 나쁘다"의 경우 BoW는 단어 빈도만 체크함. -> Transformer(BERT, GPT)가 등장함.
2) **문서 길이 편향**: 단순히 문장이 길어지면 횟수가 많아진다(큰 의미없는 단어여도 -> 긴 리뷰가 무조건 유리한 편향이 발생함.
3) **흔한 단어 과대평가**

## 2. TF - 단어의 빈도를 비율로

```
TF = 이 단어의 횟수 / 문서 전체 단어 수   (결과: 0.0 ~ 1.0)
```
--> 결과값이 0~1이라 컴퓨터가 인식하기 좋음.

--> 문서마다 단어의 개수 및 문장 길이가 다를텐데 하나의 기준 지표가 됨.

In [67]:
def TF(word, word_list):
    count    = word_list.count(word)
    total = len(word_list)
    return count / total

In [68]:
# 리뷰1: "치킨 치킨 배달 맛있다"
print(f"TF(치킨)   = 2/4 = {TF('치킨', reviews_BoW['review1']):.3f}  ← 전체의 절반!")
print(f"TF(배달)   = 1/4 = {TF('배달', reviews_BoW['review1']):.3f}")
print(f"TF(맛있다) = 1/4 = {TF('맛있다', reviews_BoW['review1']):.3f}")
print(f"TF(피자)   = 0/4 = {TF('피자', reviews_BoW['review1']):.3f}  ← 등장 안 함")

TF(치킨)   = 2/4 = 0.500  ← 전체의 절반!
TF(배달)   = 1/4 = 0.250
TF(맛있다) = 1/4 = 0.250
TF(피자)   = 0/4 = 0.000  ← 등장 안 함


In [69]:
# TF가 문서 길이 편향을 어떻게 해결하는지 확인
r1 = "치킨 맛있다"
r2 = "치킨 치킨 배달 맛있다 빠르다 좋다"

print("짧은 리뷰: ",r1)
print("긴 리뷰: ",r2)
r1 = "치킨 맛있다".split()
r2 = "치킨 치킨 배달 맛있다 빠르다 좋다".split()

print("\n[BoW 횟수 비교]")
print(f"  짧은 리뷰 '치킨': {r1.count('치킨')}번")
print(f"  긴   리뷰 '치킨': {r2.count('치킨')}번  ← 긴 리뷰가 더 많아 보임 ")

print()
print("[TF 비율 비교]")
print(f"  짧은 리뷰 '치킨' TF: {TF('치킨', r1):.3f}  ({r1.count('치킨')}/{len(r1)})")
print(f"  긴   리뷰 '치킨' TF: {TF('치킨', r2):.3f}  ({r2.count('치킨')}/{len(r2)}). ← 하지만 비율로 비교하면 더 낮음")
print()

짧은 리뷰:  치킨 맛있다
긴 리뷰:  치킨 치킨 배달 맛있다 빠르다 좋다

[BoW 횟수 비교]
  짧은 리뷰 '치킨': 1번
  긴   리뷰 '치킨': 2번  ← 긴 리뷰가 더 많아 보임 

[TF 비율 비교]
  짧은 리뷰 '치킨' TF: 0.500  (1/2)
  긴   리뷰 '치킨' TF: 0.333  (2/6). ← 하지만 비율로 비교하면 더 낮음



In [70]:
tf_dict = {}

for name, review in reviews_BoW.items():
    tf_dict[name] = [TF(word, review) for word in words]

print("TF 행렬  [" + ", ".join(words) + "]")
print("-" * 55)
print(f"리뷰1: {[round(v, 3) for v in tf_dict['review1']]}")
print(f"리뷰2: {[round(v, 3) for v in tf_dict['review2']]}")
print(f"리뷰3: {[round(v, 3) for v in tf_dict['review3']]}")
print()
print("'배달'은 리뷰1·2에서 모두 TF=0.25 → 두 리뷰를 구별 못 함 → IDF로 해결!")

TF 행렬  [맛있다, 배달, 빠르다, 서비스, 치킨, 피자, 좋다]
-------------------------------------------------------
리뷰1: [0.25, 0.25, 0.0, 0.0, 0.5, 0.0, 0.0]
리뷰2: [0.0, 0.25, 0.25, 0.0, 0.25, 0.25, 0.0]
리뷰3: [0.0, 0.0, 0.0, 0.25, 0.0, 0.5, 0.25]

'배달'은 리뷰1·2에서 모두 TF=0.25 → 두 리뷰를 구별 못 함 → IDF로 해결!


---
## 3. IDF — 희귀한 단어일수록 중요하다

```
IDF = log( 전체 문서 수 / (이 단어가 등장한 문서 수 + 1) )
```
- 모든 문서에 나오는 단어 → 분모 커짐 → IDF ≈ 0 (중요하지 않음)  
- 일부 문서에만 나오는 단어 → 분모 작음 → IDF 높음 (중요)  
- `+1`: 분모가 0이 되는 경우를 방지 (등장 문서가 0개인 새 단어 처리)

In [71]:
df_dict = {}

for word in words:
    df_dict[word] = sum(1 if word in review else 0 for review in reviews_BoW.values())

print("등장 문서 수 (전체 3개 중):")
for word, df in df_dict.items():
    print(f"  {word}: {df}개  ({'■' * df})")

등장 문서 수 (전체 3개 중):
  맛있다: 1개  (■)
  배달: 2개  (■■)
  빠르다: 1개  (■)
  서비스: 1개  (■)
  치킨: 2개  (■■)
  피자: 2개  (■■)
  좋다: 1개  (■)


> **결과 해설**  
> - '배달', '치킨', '피자'는 각각 2개 리뷰에 등장합니다 → 상대적으로 흔한 편  
> - '맛있다', '빠르다', '서비스', '좋다'는 1개 리뷰에만 등장합니다 → 더 희귀  
>
> 등장 문서 수가 적을수록 IDF가 높아져서 해당 리뷰를 더 잘 대표하는 단어로 평가됩니다.

In [72]:
N = 3

idf_dict = {word: math.log(N / (df + 1)) for word, df in df_dict.items()}

print("IDF 값 (높을수록 희귀한 단어):")
for word, df in df_dict.items():
    idf = idf_dict[word]

    comment = ""
    if df == 1:
        comment = "← 1개 리뷰에만 등장 (희귀)"
    elif df == 2:
        comment = "← 2개 리뷰에 등장 (흔함)"

    print(f"  {word}: log({N}/{df+1}) = {idf:.3f}  {comment}")

IDF 값 (높을수록 희귀한 단어):
  맛있다: log(3/2) = 0.405  ← 1개 리뷰에만 등장 (희귀)
  배달: log(3/3) = 0.000  ← 2개 리뷰에 등장 (흔함)
  빠르다: log(3/2) = 0.405  ← 1개 리뷰에만 등장 (희귀)
  서비스: log(3/2) = 0.405  ← 1개 리뷰에만 등장 (희귀)
  치킨: log(3/3) = 0.000  ← 2개 리뷰에 등장 (흔함)
  피자: log(3/3) = 0.000  ← 2개 리뷰에 등장 (흔함)
  좋다: log(3/2) = 0.405  ← 1개 리뷰에만 등장 (희귀)


> **결과 해설**  
> - 2개 리뷰에 등장하는 '배달', '치킨', '피자'는 IDF가 낮고,  
> - 1개 리뷰에만 등장하는 '맛있다', '빠르다', '서비스', '좋다'는 IDF가 높다.  
>
> **IDF가 높다 = 이 단어가 등장하면 어떤 리뷰인지 특정하기 쉽다** 는 의미이다.  


---
## 4. TF-IDF

```
TF-IDF = TF × IDF
       = "이 문서에서 자주 나왔고" × "전체에서 희귀한 단어"
       = 이 문서의 진짜 핵심어 점수

In [73]:
# TfidfVectorizer: TF-IDF 계산을 자동으로 해주는 sklearn 도구
vectorizer = TfidfVectorizer()

# fit_transform(): 두 가지를 동시에 수행
#   fit     → 어휘 목록 만들기 + 각 단어의 IDF 계산 (학습)
#   transform → 각 리뷰를 TF-IDF 벡터로 변환
tfidf_mat = vectorizer.fit_transform([review1, review2, review3])

# get_feature_names_out(): 학습된 어휘 목록 확인
voca = vectorizer.get_feature_names_out()

print("어휘:", voca)
print("행렬 크기:", tfidf_mat.shape, " ← (리뷰 수 × 단어 수)")

어휘: ['맛있다' '배달' '빠르다' '서비스' '좋다' '치킨' '피자']
행렬 크기: (3, 7)  ← (리뷰 수 × 단어 수)


In [74]:
print(f"리뷰1: '{' '.join(reviews_BoW['review1'])}'")
print()
print(f"{'단어':8}  TF  ×  IDF  =  TF-IDF")
print("-" * 38)

tfidf_review1 = {}

for word, tf, idf in zip(words, tf_dict["review1"], [idf_dict[w] for w in words]):
    tfidf = tf * idf
    tfidf_review1[word] = tfidf

    print(f"{word:8}  {tf:.3f} × {idf:.3f} = {tfidf:.4f}")

print()
top_word = max(tfidf_review1, key=tfidf_review1.get)
print(f"→ 리뷰1 핵심어: '{top_word}' ({tfidf_review1[top_word]:.4f})")

리뷰1: '치킨 치킨 배달 맛있다'

단어        TF  ×  IDF  =  TF-IDF
--------------------------------------
맛있다       0.250 × 0.405 = 0.1014
배달        0.250 × 0.000 = 0.0000
빠르다       0.000 × 0.405 = 0.0000
서비스       0.000 × 0.405 = 0.0000
치킨        0.500 × 0.000 = 0.0000
피자        0.000 × 0.000 = 0.0000
좋다        0.000 × 0.405 = 0.0000

→ 리뷰1 핵심어: '맛있다' (0.1014)


> **결과 해설**  
> '치킨'은 TF가 0.500으로 가장 높고(2번 등장), IDF도 적당히 있어서 TF-IDF가 가장 높다.  
> '맛있다'는 TF는 0.250이지만 IDF가 더 높아서 두 번째로 높은 점수이다.  
> '배달'은 TF가 0.250이지만 2개 리뷰에 나오는 흔한 단어라 IDF가 낮아 점수가 낮다.  
>
> → **TF-IDF는 "이 리뷰에서 자주 나왔으면서, 다른 리뷰에는 별로 없는 단어"에 높은 점수를 준다.**

In [75]:
# .toarray(): 희소 행렬(메모리 절약형) → 일반 숫자 배열로 변환
mat = tfidf_mat.toarray()

print("TF-IDF 행렬:")
print(f"{'':8}" + "  ".join(f"{w:7}" for w in voca))
print("-" * 65)
print(f"리뷰1:  " + "  ".join(f"{v:.4f}" for v in mat[0]))
print(f"리뷰2:  " + "  ".join(f"{v:.4f}" for v in mat[1]))
print(f"리뷰3:  " + "  ".join(f"{v:.4f}" for v in mat[2]))

r1_top = voca[mat[0].argmax()]
r2_top = voca[mat[1].argmax()]
r3_top = voca[mat[2].argmax()]

print()
print(f"리뷰1 핵심어: '{r1_top}'")
print(f"리뷰2 핵심어: '{r2_top}'")
print(f"리뷰3 핵심어: '{r3_top}'")

TF-IDF 행렬:
        맛있다      배달       빠르다      서비스      좋다       치킨       피자     
-----------------------------------------------------------------
리뷰1:  0.5069  0.3855  0.0000  0.0000  0.0000  0.7710  0.0000
리뷰2:  0.0000  0.4599  0.6047  0.0000  0.0000  0.4599  0.4599
리뷰3:  0.0000  0.0000  0.0000  0.4815  0.4815  0.0000  0.7324

리뷰1 핵심어: '치킨'
리뷰2 핵심어: '빠르다'
리뷰3 핵심어: '피자'


---
## 5. 코사인 유사도 — 얼마나 비슷한가?

```
코사인 유사도 = 두 벡터가 얼마나 같은 방향을 가리키는가
              0.0 (완전히 다름)  ~  1.0 (완전히 같음)
```
길이(문서 분량)는 무시하고 **방향(어떤 단어를 쓰는가)** 만 비교한다.

## 6.복습과제

In [82]:
기사1 = "삼성전자가 AI 스마트폰 신제품을 공개했습니다. 카메라 성능과 배터리 효율이 향상됐습니다."
기사2 = "애플이 AI 기능을 탑재한 스마트폰 업데이트를 발표했습니다. 카메라 기능과 배터리 성능도 개선됐습니다."
기사3 = "삼성전자가 반도체 투자 계획을 발표했습니다. AI 반도체와 메모리 시장 경쟁력이 강화될 전망입니다."
기사4 = "SK하이닉스가 AI 반도체 생산을 확대합니다. 메모리 수요 증가에 대응하기 위한 투자도 이어집니다."
기사5 = "국내 스타트업이 AI 의료 진단 서비스를 출시했습니다. 병원 데이터 분석과 진단 정확도 향상이 핵심입니다."
기사6 = "국내 헬스케어 기업이 의료 AI 플랫폼을 공개했습니다. 데이터 분석 기반으로 진단 속도와 정확도를 높였습니다."

print("기사1 [스마트폰] :", 기사1[:40], "...")
print("기사2 [스마트폰] :", 기사2[:40], "...")
print("기사3 [반도체]  :", 기사3[:40], "...")
print("기사4 [반도체]  :", 기사4[:40], "...")
print("기사5 [의료 AI] :", 기사5[:40], "...")
print("기사6 [의료 AI] :", 기사6[:40], "...")

기사1 [스마트폰] : 삼성전자가 AI 스마트폰 신제품을 공개했습니다. 카메라 성능과 배터리 효 ...
기사2 [스마트폰] : 애플이 AI 기능을 탑재한 스마트폰 업데이트를 발표했습니다. 카메라 기능 ...
기사3 [반도체]  : 삼성전자가 반도체 투자 계획을 발표했습니다. AI 반도체와 메모리 시장  ...
기사4 [반도체]  : SK하이닉스가 AI 반도체 생산을 확대합니다. 메모리 수요 증가에 대응하 ...
기사5 [의료 AI] : 국내 스타트업이 AI 의료 진단 서비스를 출시했습니다. 병원 데이터 분석 ...
기사6 [의료 AI] : 국내 헬스케어 기업이 의료 AI 플랫폼을 공개했습니다. 데이터 분석 기반 ...


In [78]:
# STEP 1: 전처리 — 6개 기사 각각 명사 추출
def 전처리(문장):
  noun = okt.nouns(문장)
  return noun

처리_기사1 = 전처리(기사1)
처리_기사2 = 전처리(기사2)
처리_기사3 = 전처리(기사3)
처리_기사4 = 전처리(기사4)
처리_기사5 = 전처리(기사5)
처리_기사6 = 전처리(기사6)

print("전처리 결과 (명사만 추출):")
print(f"  기사1: {처리_기사1}")
print(f"  기사2: {처리_기사2}")
print(f"  기사3: {처리_기사3}")
print(f"  기사4: {처리_기사4}")
print(f"  기사5: {처리_기사5}")
print(f"  기사6: {처리_기사6}")

전처리 결과 (명사만 추출):
  기사1: ['전자', '스마트폰', '신제품', '공개', '카메라', '성능', '배터리', '효율', '향상']
  기사2: ['애플', '기능', '탑재', '스마트폰', '업데이트', '발표', '카메라', '기능', '배터리', '성능', '개선']
  기사3: ['전자', '반도체', '투자', '계획', '발표', '반도체', '메모리', '시장', '경쟁력', '강화', '전망']
  기사4: ['하이닉스', '반도체', '생산', '확대', '메모리', '수요', '증가', '대응', '위', '투자']
  기사5: ['국내', '스타트업', '의료', '진단', '서비스', '출시', '병원', '데이터', '분석', '진단', '정확도', '향상', '핵심']
  기사6: ['국내', '헬', '스케', '기업', '의료', '플랫폼', '공개', '데이터', '분석', '기반', '진단', '속도', '정확도']


In [79]:
# STEP 2: TF-IDF 행렬 생성
# 전처리 결과(리스트)를 ' '.join()으로 문자열로 변환 후 vectorizer에 전달
기사_vec = TfidfVectorizer()
기사_tfidf = 기사_vec.fit_transform([
    ' '.join(처리_기사1), ' '.join(처리_기사2),
    ' '.join(처리_기사3), ' '.join(처리_기사4),
    ' '.join(처리_기사5), ' '.join(처리_기사6)
])
기사_어휘 = 기사_vec.get_feature_names_out()
기사_배열 = 기사_tfidf.toarray()

print(f"전체 어휘: {len(기사_어휘)}개")
print(f"행렬 크기: {기사_tfidf.shape}  ← (기사 6개 × 단어 수)")

전체 어휘: 45개
행렬 크기: (6, 45)  ← (기사 6개 × 단어 수)


In [84]:
# STEP 3: 기사별 핵심어 (.argmax()로 TF-IDF 최고 단어 추출)
print("기사별 핵심어:")
print(f"  기사1 [스마트폰]: '{기사_어휘[기사_배열[0].argmax()]}'  ({기사_배열[0].max():.3f})")
print(f"  기사2 [스마트폰]: '{기사_어휘[기사_배열[1].argmax()]}'  ({기사_배열[1].max():.3f})")
print(f"  기사3 [반도체]:   '{기사_어휘[기사_배열[2].argmax()]}'  ({기사_배열[2].max():.3f})")
print(f"  기사4 [반도체]:   '{기사_어휘[기사_배열[3].argmax()]}'  ({기사_배열[3].max():.3f})")
print(f"  기사5 [의료 AI]:  '{기사_어휘[기사_배열[4].argmax()]}'  ({기사_배열[4].max():.3f})")
print(f"  기사6 [의료 AI]:  '{기사_어휘[기사_배열[5].argmax()]}'  ({기사_배열[5].max():.3f})")

기사별 핵심어:
  기사1 [스마트폰]: '신제품'  (0.386)
  기사2 [스마트폰]: '기능'  (0.593)
  기사3 [반도체]:   '반도체'  (0.509)
  기사4 [반도체]:   '대응'  (0.353)
  기사5 [의료 AI]:  '진단'  (0.479)
  기사6 [의료 AI]:  '기반'  (0.321)


In [85]:
# STEP 4: 유사도 행렬 — 같은 주제 기사끼리 높게 나오면 성공!
기사_유사도 = cosine_similarity(기사_tfidf)

print("유사도 확인:")
print()
# 같은 주제 쌍: 높아야 함
print(f"  [같은 주제] 스마트폰: 기사1 ↔ 기사2 = {기사_유사도[0][1]:.3f}  {' 높음!' if 기사_유사도[0][1] > 0.1 else ' 낮음'}")
print(f"  [같은 주제] 반도체: 기사3 ↔ 기사4 = {기사_유사도[2][3]:.3f}  {' 높음!' if 기사_유사도[2][3] > 0.1 else ' 낮음'}")
print(f"  [같은 주제] 의료 AI: 기사5 ↔ 기사6 = {기사_유사도[4][5]:.3f}  {' 높음!' if 기사_유사도[4][5] > 0.1 else ' 낮음'}")
print()
# 다른 주제 쌍: 낮아야 함
print(f"  [다른 주제] 기사1(스마트폰) ↔ 기사3(반도체) = {기사_유사도[0][2]:.3f}  {' 낮음!' if 기사_유사도[0][2] < 0.1 else ' 높음'}")
print(f"  [다른 주제] 기사1(스마트폰) ↔ 기사5(의료 AI)  = {기사_유사도[0][4]:.3f}  {' 낮음!' if 기사_유사도[0][4] < 0.1 else ' 높음'}")
print()
print("같은 주제끼리 유사도 높고, 다른 주제끼리 낮으면 성공! ")

유사도 확인:

  [같은 주제] 스마트폰: 기사1 ↔ 기사2 = 0.308   높음!
  [같은 주제] 반도체: 기사3 ↔ 기사4 = 0.295   높음!
  [같은 주제] 의료 AI: 기사5 ↔ 기사6 = 0.441   높음!

  [다른 주제] 기사1(스마트폰) ↔ 기사3(반도체) = 0.081   낮음!
  [다른 주제] 기사1(스마트폰) ↔ 기사5(의료 AI)  = 0.076   낮음!

같은 주제끼리 유사도 높고, 다른 주제끼리 낮으면 성공! 🎯


### 핵심 코드
```python
vectorizer = TfidfVectorizer()
tfidf      = vectorizer.fit_transform(["단어1 단어2", "단어3 단어4"])
유사도     = cosine_similarity(tfidf)